# Segment 5 - Data Quality & Order Integrity Analysis

## Analysis 13 — Customer-Order Referential Integrity Analysis
### Business question: 
### Which orders reference customers who do not exist in the customer master table, including orders with missing customer IDs?

In [0]:
SELECT
    o.order_id,
    o.customer_id,
    o.order_date,
    o.order_status,
    o.order_amount,
    CASE
        WHEN o.customer_id IS NULL
            THEN 'Missing Customer ID'
        ELSE 'Customer Not Found'
    END AS issue_type
FROM retailops360.source.orders AS o
LEFT ANTI JOIN retailops360.source.customers AS c
    ON o.customer_id = c.customer_id
ORDER BY o.order_date, o.order_id;

order_id,customer_id,order_date,order_status,order_amount,issue_type


## Analysis 14 — Orphan Payment Transaction Analysis
### Business question: 
### Which payment transactions cannot be matched to a valid order, and what is their total monetary value?

In [0]:
WITH orphan_payments AS (
    SELECT
        p.payment_id,
        p.order_id,
        p.payment_amount,
        p.payment_status,
        CASE
            WHEN p.order_id IS NULL
                THEN 'Missing Order ID'
            ELSE 'Order Not Found'
        END AS issue_type
    FROM retailops360.source.payments AS p
    LEFT JOIN retailops360.source.orders AS o
        ON p.order_id = o.order_id
    WHERE o.order_id IS NULL
)

SELECT
    payment_id,
    order_id,
    payment_amount,
    payment_status,
    issue_type,
    COUNT(*) OVER () AS total_orphan_payments,
    ROUND(
        SUM(payment_amount) OVER (),
        2
    ) AS total_orphan_payment_amount
FROM orphan_payments
ORDER BY payment_amount DESC, payment_id;

payment_id,order_id,payment_amount,payment_status,issue_type,total_orphan_payments,total_orphan_payment_amount


## Analysis 15 — Complete Order-to-Payment Integrity Reconciliation
### Business question: 
### Which orders and payment records fail to reconcile across both systems, including unmatched records and amount discrepancies?

In [0]:
WITH payment_summary AS (
    SELECT
        order_id,
        ROUND(
            SUM(
                CASE
                    WHEN payment_status = 'Captured'
                        THEN payment_amount
                    ELSE 0
                END
            ),
            2
        ) AS captured_amount,
        COUNT(payment_id) AS payment_count
    FROM retailops360.source.payments
    GROUP BY order_id
),

reconciliation AS (
    SELECT
        o.order_id AS order_id,
        p.order_id AS payment_order_id,
        o.order_amount,
        p.captured_amount,
        p.payment_count
    FROM retailops360.source.orders AS o
    FULL OUTER JOIN payment_summary AS p
        ON o.order_id = p.order_id
)

SELECT
    COALESCE(order_id, payment_order_id)
        AS reference_order_id,

    order_amount,

    COALESCE(captured_amount, 0)
        AS captured_amount,

    COALESCE(payment_count, 0)
        AS payment_count,

    ROUND(
        COALESCE(order_amount, 0)
        - COALESCE(captured_amount, 0),
        2
    ) AS amount_difference,

    CASE
        WHEN order_id IS NULL
            THEN 'Orphan Payment'

        WHEN payment_order_id IS NULL
            THEN 'No Payment Record'

        WHEN captured_amount = 0
            THEN 'No Captured Payment'

        WHEN captured_amount < order_amount
            THEN 'Underpaid'

        WHEN captured_amount > order_amount
            THEN 'Overpaid'

        ELSE 'Fully Reconciled'
    END AS reconciliation_status

FROM reconciliation

WHERE order_id IS NULL
   OR payment_order_id IS NULL
   OR captured_amount <> order_amount

ORDER BY reconciliation_status, reference_order_id;

reference_order_id,order_amount,captured_amount,payment_count,amount_difference,reconciliation_status
16,2354.00,0.00,0,2354.00,No Payment Record
17,3184.00,0.00,0,3184.00,No Payment Record
29,1678.00,0.00,0,1678.00,No Payment Record
33,5215.00,0.00,0,5215.00,No Payment Record
34,2218.00,0.00,0,2218.00,No Payment Record
40,810.00,0.00,0,810.00,No Payment Record
65,1835.00,0.00,0,1835.00,No Payment Record
72,5139.00,0.00,0,5139.00,No Payment Record
73,4150.00,0.00,0,4150.00,No Payment Record
84,691.00,0.00,0,691.00,No Payment Record
